In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [2]:
RESULTS_DIR = "../evaluation/results"
IMAGES_DIR = "./images"
Path(IMAGES_DIR).mkdir(parents=True, exist_ok=True)

os.listdir(RESULTS_DIR)

['phi4_latest_sync_chain_results_validation.csv',
 'llama3_2_latest_sync_chain_results_validation.csv',
 'phi4_latest_async_chain_results_validation.csv',
 'llama3_2_latest_async_chain_results_validation.csv']

In [3]:
def load_benchmark_df(results_dir: str) -> pd.DataFrame:
    benchmark_df = pd.DataFrame()
    for filename in os.listdir(results_dir):
        df = pd.read_csv(os.path.join(results_dir, filename))
        model_name = "_".join(filename.split("/")[-1].split("_")[0:2])
        df["ModelName"] = model_name
        is_async = "async" in filename
        df["Algorithm"] = "Async" if is_async else "Sync"
        benchmark_df = pd.concat([benchmark_df, df])
    return benchmark_df

benchmark_df = load_benchmark_df(RESULTS_DIR)
benchmark_df.head()

,RowID,ErrorOccurred,ExecutionOutput,HasTimedOut,ExecutionError,ExecutionTime,ExpectedOutput,ActualOutput,CorrectOutput,FirstOutput,ModelName,Algorithm
0,520,False,120,False,NaN,8.329806,120.0,120,True,8.329044,phi4_latest,Sync
1,522,False,6,False,NaN,16.821880,6.0,6,True,16.821420,phi4_latest,Sync
2,522,False,5,False,NaN,15.817442,5.0,5,True,15.816990,phi4_latest,Sync
3,524,False,22,False,NaN,10.937061,22.0,22,True,10.936627,phi4_latest,Sync
4,524,False,10,False,NaN,10.806910,10.0,10,True,10.806437,phi4_latest,Sync


In [4]:
def display_execution_time_describe(df: pd.DataFrame) -> None:
    print(df.groupby(["ModelName", "Algorithm"])["ExecutionTime"].describe())

display_execution_time_describe(benchmark_df)

                       count       mean       std       min        25%  \
ModelName   Algorithm                                                    
llama3_2    Async        5.0   4.117500  1.262106  2.605744   3.113175   
            Sync         5.0   4.107834  1.269905  2.568893   3.119397   
phi4_latest Async        5.0  12.590209  3.795676  8.320115  10.665960   
            Sync         5.0  12.542620  3.618523  8.329806  10.806910   

                             50%        75%        max  
ModelName   Algorithm                                   
llama3_2    Async       4.227390   5.028514   5.612676  
            Sync        4.227306   5.001948   5.621627  
phi4_latest Async      10.757593  16.361130  16.846246  
            Sync       10.937061  15.817442  16.821880  


In [5]:
def save_summary_tables(df: pd.DataFrame, tables_dir: str) -> None:
    Path(tables_dir).mkdir(parents=True, exist_ok=True)
    grouped = df.groupby(["ModelName", "Algorithm", "ErrorOccurred"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    grouped.to_csv(os.path.join(tables_dir, "summary_by_error.csv"), index=False)

save_summary_tables(benchmark_df, "./tables")

def display_summary_tables(df: pd.DataFrame) -> None:
    grouped = df.groupby(["ModelName", "Algorithm", "ErrorOccurred"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    non_error_df = df[df["ErrorOccurred"] == False]
    grouped_correct = non_error_df.groupby(["ModelName", "Algorithm", "CorrectOutput"]).agg(
        Count=("ExecutionTime", "count"),
        Avg_ExecTime=("ExecutionTime", "mean"),
        Median_ExecTime=("ExecutionTime", "median"),
        Avg_FirstOutput=("FirstOutput", "mean"),
        Median_FirstOutput=("FirstOutput", "median")
    ).reset_index()
    error_summary = df.groupby(["ModelName", "Algorithm"]).agg(
        ErrorCount=("ErrorOccurred", lambda x: (x == True).sum())
    ).reset_index()
    final = pd.merge(grouped, error_summary, on=["ModelName", "Algorithm"], how="left")
    print("Summary by ModelName, Algorithm, and ErrorOccurred:")
    display(final)
    print("\nSummary by ModelName, Algorithm, and CorrectOutput (non-error cases only):")
    display(grouped_correct)

display_summary_tables(benchmark_df)

Summary by ModelName, Algorithm, and ErrorOccurred:


,ModelName,Algorithm,ErrorOccurred,Count,Avg_ExecTime,Median_ExecTime,Avg_FirstOutput,Median_FirstOutput,ErrorCount
0,llama3_2,Async,False,4,3.889746,3.670282,2.822348,2.711047,1
1,llama3_2,Async,True,1,5.028514,5.028514,2.542054,2.542054,1
2,llama3_2,Sync,False,4,3.884306,3.673351,3.883775,3.672880,1
3,llama3_2,Sync,True,1,5.001948,5.001948,5.001948,5.001948,1
4,phi4_latest,Async,False,5,12.590209,10.757593,8.072732,8.599195,0
5,phi4_latest,Sync,False,5,12.542620,10.937061,12.542104,10.936627,0



Summary by ModelName, Algorithm, and CorrectOutput (non-error cases only):


,ModelName,Algorithm,CorrectOutput,Count,Avg_ExecTime,Median_ExecTime,Avg_FirstOutput,Median_FirstOutput
0,llama3_2,Async,False,1,2.605744,2.605744,1.872706,1.872706
1,llama3_2,Async,True,3,4.317747,4.227390,3.138896,2.933337
2,llama3_2,Sync,False,1,2.568893,2.568893,2.568191,2.568191
3,llama3_2,Sync,True,3,4.322777,4.227306,4.322303,4.226786
4,phi4_latest,Async,True,5,12.590209,10.757593,8.072732,8.599195
5,phi4_latest,Sync,True,5,12.542620,10.937061,12.542104,10.936627


In [6]:
def plot_execution_time_catplot(df: pd.DataFrame, image_path: str) -> None:
    sns.set_style(style="whitegrid")
    g_exec = sns.catplot(
        data=df, 
        x="Algorithm", 
        y="ExecutionTime", 
        hue="ErrorOccurred", 
        col="ModelName", 
        kind="box",
        height=4, 
        aspect=0.8,
        palette="Set2",
        sharey=False
    )
    g_exec.set_titles("Model: {col_name}")
    g_exec.figure.suptitle("Execution Time by Algorithm and Error Occurrence", y=1.05)
    g_exec.set_axis_labels("Algorithm", "Execution Time (s)")
    plt.tight_layout()
    g_exec.savefig(image_path)
    plt.close()

def plot_first_output_catplot(df: pd.DataFrame, image_path: str) -> None:
    sns.set_style(style="whitegrid")
    g_first = sns.catplot(
        data=df, 
        x="Algorithm", 
        y="FirstOutput", 
        hue="ErrorOccurred", 
        col="ModelName", 
        kind="box",
        height=4, 
        aspect=0.8,
        palette="Set2",
        sharey=False
    )
    g_first.set_titles("Model: {col_name}")
    g_first.figure.suptitle("First Output Time by Algorithm and Error Occurrence", y=1.05)
    g_first.set_axis_labels("Algorithm", "First Output Time (s)")
    plt.tight_layout()
    g_first.savefig(image_path)
    plt.close()

plot_execution_time_catplot(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_catplot.png"))
plot_first_output_catplot(benchmark_df, os.path.join(IMAGES_DIR, "first_output_catplot.png"))

In [7]:
def plot_execution_time_hist(df: pd.DataFrame, image_path: str) -> None:
    plt.figure(figsize=(10, 6))
    async_df = df[df["Algorithm"] == "Async"]
    sync_df = df[df["Algorithm"] == "Sync"]
    sns.histplot(x=async_df["ExecutionTime"], bins=20, kde=True, label="Async")
    sns.histplot(x=sync_df["ExecutionTime"], bins=20, kde=True, label="Sync")
    plt.title("Distribution of Execution Times")
    plt.xlabel("Execution Time (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_execution_time_hist(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_hist.png"))

In [8]:
def plot_first_output_hist(df: pd.DataFrame, image_path: str) -> None:
    plt.figure(figsize=(10, 6))
    async_df = df[df["Algorithm"] == "Async"]
    sync_df = df[df["Algorithm"] == "Sync"]
    sns.histplot(x=async_df["FirstOutput"], bins=20, kde=True, label="Async")
    sns.histplot(x=sync_df["FirstOutput"], bins=20, kde=True, label="Sync")
    plt.title("Distribution of First Output Times")
    plt.xlabel("First Output Time (seconds)")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_first_output_hist(benchmark_df, os.path.join(IMAGES_DIR, "first_output_hist.png"))

In [9]:
def plot_execution_time_boxplot(df: pd.DataFrame, image_path: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    df.boxplot(column="ExecutionTime", by=["ErrorOccurred", "Algorithm"], ax=ax)
    plt.title("Execution Time by Error Occurrence and Algorithm")
    plt.suptitle("")
    plt.xlabel("Error Occurred and Algorithm")
    plt.ylabel("Execution Time (seconds)")
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_execution_time_boxplot(benchmark_df, os.path.join(IMAGES_DIR, "execution_time_boxplot.png"))

In [ ]:
def compute_and_display_delta_table(df: pd.DataFrame, tables_dir: str) -> None:
    # Pivot data to get Async and Sync execution times for the same RowID
    pivot_df = df.pivot_table(
        index=["RowID", "ModelName"],
        columns="Algorithm",
        values=["ExecutionTime", "FirstOutput", "ErrorOccurred", "CorrectOutput"],
        aggfunc="first"
    ).reset_index()
    
    # Flatten column names
    pivot_df.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in pivot_df.columns.values]
    
    # Compute delta (Async - Sync) for ExecutionTime and FirstOutput
    pivot_df["Delta_ExecutionTime"] = pivot_df["ExecutionTime_Async"] - pivot_df["ExecutionTime_Sync"]
    pivot_df["Delta_FirstOutput"] = pivot_df["FirstOutput_Async"] - pivot_df["FirstOutput_Sync"]
    
    # Create summary table
    Path(tables_dir).mkdir(parents=True, exist_ok=True)
    summary = pivot_df.groupby("ModelName").agg(
        Count=("Delta_ExecutionTime", "count"),
        Avg_Delta_ExecTime=("Delta_ExecutionTime", "mean"),
        Median_Delta_ExecTime=("Delta_ExecutionTime", "median"),
        Std_Delta_ExecTime=("Delta_ExecutionTime", "std"),
        Min_Delta_ExecTime=("Delta_ExecutionTime", "min"),
        Max_Delta_ExecTime=("Delta_ExecutionTime", "max"),
        Avg_Delta_FirstOutput=("Delta_FirstOutput", "mean"),
        Median_Delta_FirstOutput=("Delta_FirstOutput", "median")
    ).reset_index()
    summary.to_csv(os.path.join(tables_dir, "summary_delta.csv"), index=False)
    
    print("Summary of Delta (Async - Sync) by ModelName:")
    display(summary)

    delta_df = compute_and_display_delta_table(benchmark_df, "./tables")

    return pivot_df

Summary of Delta (ExecutionTime - FirstOutput) by ModelName, Algorithm, and ErrorOccurred:


,ModelName,Algorithm,ErrorOccurred,Count,Avg_Delta,Median_Delta,Std_Delta,Min_Delta,Max_Delta
0,llama3_2,Async,False,4,1.067398,0.678728,1.095892,0.232797,2.679339
1,llama3_2,Async,True,1,2.486461,2.486461,NaN,2.486461,2.486461
2,llama3_2,Sync,False,4,0.000530,0.000498,0.000121,0.000423,0.000702
3,llama3_2,Sync,True,1,0.000000,0.000000,NaN,0.000000,0.000000
4,phi4_latest,Async,False,5,4.517476,2.853040,3.521183,1.525304,8.411157
5,phi4_latest,Sync,False,5,0.000516,0.000460,0.000138,0.000434,0.000762



Summary of Delta for non-error cases by CorrectOutput:


,ModelName,Algorithm,CorrectOutput,Count,Avg_Delta,Median_Delta,Std_Delta
0,llama3_2,Async,False,1,0.733039,0.733039,NaN
1,llama3_2,Async,True,3,1.178851,0.624418,1.314130
2,llama3_2,Sync,False,1,0.000702,0.000702,NaN
3,llama3_2,Sync,True,3,0.000473,0.000477,0.000048
4,phi4_latest,Async,True,5,4.517476,2.853040,3.521183
5,phi4_latest,Sync,True,5,0.000516,0.000460,0.000138


In [ ]:
def plot_delta_boxplot(delta_df: pd.DataFrame, image_path: str) -> None:
    # Create box plot for delta execution time by model
    sns.set_style(style="whitegrid")
    plt.figure(figsize=(12, 6))
    sns.boxplot(
        data=delta_df,
        x="ModelName",
        y="Delta_ExecutionTime",
        palette="Set2"
    )
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5, label='No difference')
    plt.title("Delta ExecutionTime (Async - Sync) by Model")
    plt.xlabel("Model Name")
    plt.ylabel("Delta ExecutionTime (s)\n(Negative = Async faster)")
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.savefig(image_path)
    plt.close()

plot_delta_boxplot(delta_df, os.path.join(IMAGES_DIR, "delta_execution_time_boxplot.png"))